El objetivo actual del proyecto es crear un modelo que en base a señales edf aprenda a predecir si el paciente esta teniendo un sueño agradable o desagradable.
Como esto es un poco abstracto en caso de no obtener resultados me centraria en otro objetivo distinto un poco mas simple, entrenar un modelo para distinguir entre sueño vs no sueño en base al edf.

In [ ]:
import os
import mne
import yasa
import pandas as pd
import numpy as np
import shutil
import matplotlib.pyplot as plt 
from pathlib import Path

In [ ]:
df=pd.read_csv("REM_Turku/Records.csv")

df.shape

(134, 19)

In [29]:
df.head()

,Filename,Case ID,Subject ID,Experience,Treatment group,Duration,EEG sample rate,Number of EEG channels,Last sleep stage,Has EOG,Has EMG,Has ECG,Proportion artifacts,Time of awakening,Subject age,Subject sex,Subject healthy,Has more data,Remarks
0,case01_s10.edf,1,10,1,NaN,120,500,24,5,1,1,0,NaN,4:47 AM,31,1,1,0,1_2_No data for Night 1 Awakening 1
1,case02_s10.edf,2,10,2,NaN,120,500,24,5,1,1,0,NaN,5:51 AM,31,1,1,1,1_3
2,case03_s10.edf,3,10,2,NaN,120,500,24,5,1,1,0,NaN,6:47 AM,31,1,1,1,1_4
3,case04_s10.edf,4,10,2,NaN,120,500,24,5,1,1,0,NaN,7:43 AM,31,1,1,1,1_5
4,case05_s10.edf,5,10,2,NaN,120,500,24,5,1,1,0,NaN,1:38 AM,31,1,1,1,2_1


Extraigo las señales ecg, falta procesarlas y extraer caracteristicas

In [48]:
def cargar_registro(path ):

    raw = mne.io.read_raw_edf(path, preload=True) 
    raw.filter(0.3, 49)  
    data = raw.get_data() * 1e6 #microVolts (porque mne trabaja en V) 
     

 
    return data 

In [52]:
import os
import re
from pathlib import Path

ruta = Path("REM_Turku/Data/PSG")

res = []
X=[]
for archivo in ruta.glob("*.edf"):
    
    print(f"Procesando {archivo}")
    
    # Extraer case y subject
    patron = r"case(\d+)_s(\d+)"
    match = re.search(patron, archivo.stem)
    
    if match:
        case_id = int(match.group(1))
        subject_id = int(match.group(2))
        
        data = cargar_registro(archivo)
        X.append(data)
        res.append({
            "case": case_id,
            "subject": subject_id,
            "data": data
        })

Procesando REM_Turku\Data\PSG\case01_s10.edf
Procesando REM_Turku\Data\PSG\case02_s10.edf
Procesando REM_Turku\Data\PSG\case03_s10.edf
Procesando REM_Turku\Data\PSG\case04_s10.edf
Procesando REM_Turku\Data\PSG\case05_s10.edf
Procesando REM_Turku\Data\PSG\case06_s10.edf
Procesando REM_Turku\Data\PSG\case07_s10.edf
Procesando REM_Turku\Data\PSG\case08_s10.edf
Procesando REM_Turku\Data\PSG\case09_s10.edf
Procesando REM_Turku\Data\PSG\case100_s23.edf
Procesando REM_Turku\Data\PSG\case101_s23.edf
Procesando REM_Turku\Data\PSG\case102_s23.edf
Procesando REM_Turku\Data\PSG\case103_s24.edf
Procesando REM_Turku\Data\PSG\case104_s24.edf
Procesando REM_Turku\Data\PSG\case105_s24.edf
Procesando REM_Turku\Data\PSG\case106_s24.edf
Procesando REM_Turku\Data\PSG\case107_s24.edf
Procesando REM_Turku\Data\PSG\case108_s25.edf
Procesando REM_Turku\Data\PSG\case109_s25.edf
Procesando REM_Turku\Data\PSG\case10_s10.edf
Procesando REM_Turku\Data\PSG\case110_s25.edf
Procesando REM_Turku\Data\PSG\case111_s25.ed

Obtengo el overall de emociones vividas durante el sueño. En el siguiente dataframe se encuentran el numero de veces que una persona ha sentido una emocion durante el sueño. He sumado todas las veces que aparecen emociones positivas y he restado las que aparecen negativas para obtener un indice general.

In [46]:
df=pd.read_csv("REM_Turku/Data/Ratings.csv")

In [ ]:


df = pd.read_csv("REM_Turku/Data/Ratings.csv")

# Columnas de self-rating positivas
sr_pa_cols = [col for col in df.columns if col.startswith("SR_PA")]

# Columnas de self-rating negativas
sr_na_cols = [col for col in df.columns if col.startswith("SR_NA")]

# Suma positivas
sr_pa_sum = df[sr_pa_cols].sum(axis=1)

# Suma negativas
sr_na_sum = df[sr_na_cols].sum(axis=1)

# Índice de emociones
y = sr_pa_sum - sr_na_sum



0       6.0
1       2.0
2      -3.0
3      12.0
4       8.0
       ... 
117    10.0
118     8.0
119    16.0
120     5.0
121     2.0
Length: 122, dtype: float64